In [1]:
!pip install -q \
    tokenizers==0.20.3 \
    diffusers \
    transformers \
    accelerate \
    gradio \
    opencv-python-headless \
    safetensors \
    ftfy \
    pillow \
    numpy==2.1.0 \
    websockets>=13.0

!pip install -q torch --index-url https://download.pytorch.org/whl/cu121

In [3]:
import torch, cv2, gradio as gr, numpy as np, os, json, time
from PIL import Image
from diffusers import (
    StableDiffusionPipeline, StableDiffusionImg2ImgPipeline,
    StableDiffusionXLImg2ImgPipeline, StableDiffusionXLPipeline,
)
from transformers import (
    SegformerImageProcessor, SegformerForSemanticSegmentation,
    CLIPProcessor, CLIPModel,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print(f"Using device: {device}")


Using device: cuda


In [4]:
CHECKPOINT_CANDIDATES = [
    "stable-diffusion-v1-5/stable-diffusion-v1-5",  # community-maintained, ungated mirror
    "stabilityai/sd-turbo",                          # fast, ungated fallback
    "runwayml/stable-diffusion-v1-5",                # original id, kept last as a last resort
]

_working_checkpoint_cache = {}

def get_working_checkpoint(pipeline_cls=StableDiffusionImg2ImgPipeline):
    """Return the first checkpoint id in CHECKPOINT_CANDIDATES that actually loads.
    Caches the result so we only probe once per pipeline class."""
    key = pipeline_cls.__name__
    if key in _working_checkpoint_cache:
        return _working_checkpoint_cache[key]

    last_err = None
    for ckpt in CHECKPOINT_CANDIDATES:
        try:
            pipe = pipeline_cls.from_pretrained(ckpt, torch_dtype=dtype, safety_checker=None)
            del pipe
            if device == "cuda":
                torch.cuda.empty_cache()
            _working_checkpoint_cache[key] = ckpt
            print(f"[checkpoint] Using '{ckpt}' for {key}")
            return ckpt
        except Exception as e:
            last_err = e
            print(f"[checkpoint] '{ckpt}' failed to load ({e.__class__.__name__}); trying next...")
    raise RuntimeError(f"No working checkpoint found for {key}. Last error: {last_err}")


In [5]:
MODEL_MAP = {
    "Stable Diffusion 1.5 (base)": {
        "id": None,  # resolved via get_working_checkpoint()
        "pipeline_cls": StableDiffusionPipeline,
        "arch": "sd1.x",
    },
    "Realistic Vision (RealVisXL)": {
        "id": "SG161222/RealVisXL_V4.0",
        "pipeline_cls": StableDiffusionXLPipeline,   # <-- fixed: was StableDiffusionPipeline
        "arch": "sdxl",
    },
}

class StableDiffusionGenerator:
    def __init__(self):
        self.pipe = None
        self.current_model = None

    def _load_pipeline(self, model_name):
        if self.current_model == model_name and self.pipe is not None:
            return self.pipe
        cfg = MODEL_MAP[model_name]
        ckpt_id = cfg["id"] or get_working_checkpoint(cfg["pipeline_cls"])
        self.pipe = cfg["pipeline_cls"].from_pretrained(
            ckpt_id, torch_dtype=dtype, safety_checker=None
        ).to(device)
        self.current_model = model_name
        return self.pipe

    def generate(self, prompt, model_name="Stable Diffusion 1.5 (base)",
                 negative_prompt="", steps=25, guidance=7.5):
        pipe = self._load_pipeline(model_name)
        image = pipe(prompt=prompt, negative_prompt=negative_prompt,
                     num_inference_steps=steps, guidance_scale=guidance).images[0]
        return image

base_generator = StableDiffusionGenerator()


## Task 1 — Real-time Multi-Object Colorization with Semantic Segmentation


In [6]:
COLOR_MAP = {
    "building": (178, 108, 68), "tree": (34, 139, 34), "car": (200, 30, 30),
    "road": (80, 80, 80), "sky": (135, 206, 235), "person": (240, 184, 160),
    "grass": (86, 171, 47), "sidewalk": (160, 160, 160), "wall": (200, 170, 140),
    "water": (30, 100, 180),
}
SEGFORMER_ID = "nvidia/segformer-b0-finetuned-ade-512-512"

_seg_processor = SegformerImageProcessor.from_pretrained(SEGFORMER_ID)
_seg_model = SegformerForSemanticSegmentation.from_pretrained(SEGFORMER_ID).to(device)  # <-- fixed: now on GPU
_seg_model.eval()
ADE_LABELS = _seg_model.config.id2label

def segment_image(pil_image):
    inputs = _seg_processor(images=pil_image, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = _seg_model(**inputs).logits
    upsampled = torch.nn.functional.interpolate(
        logits, size=pil_image.size[::-1], mode="bilinear", align_corners=False
    )
    seg_map = upsampled.argmax(dim=1)[0].cpu().numpy()
    return seg_map  # H x W int array of ADE20K class ids

def colorize_frame(frame_bgr):
    """Segment + recolor a single BGR frame using COLOR_MAP. Runs on GPU."""
    pil_img = Image.fromarray(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
    seg_map = segment_image(pil_img)
    out = frame_bgr.copy()
    for label_name, rgb in COLOR_MAP.items():
        matching_ids = [i for i, name in ADE_LABELS.items() if label_name in name.lower()]
        for cid in matching_ids:
            mask = seg_map == cid
            if mask.any():
                out[mask] = rgb[::-1]  # RGB -> BGR
    return out

def process_video_file(video_path, progress=gr.Progress()):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 24
    w, h = int(cap.get(3)), int(cap.get(4))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    out_path = "task1_colorized_output.mp4"
    out = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

    frame_count = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        colorized = colorize_frame(frame)   # <-- fixed: every frame, no % 2 skip
        out.write(colorized)
        frame_count += 1
        if total:
            progress(frame_count / total, desc=f"Colorizing frame {frame_count}/{total}")
    cap.release()
    out.release()
    return out_path

def process_webcam_frame(frame):
    if frame is None:
        return None
    bgr = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
    colorized_bgr = colorize_frame(bgr)
    return cv2.cvtColor(colorized_bgr, cv2.COLOR_BGR2RGB)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/utils/deprecation.py:165: UserWarning: The following named arguments are not valid for `SegformerImageProcessor.__init__` and were ignored: 'feature_extractor_type'
  return func(*args, **kwargs)


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/15.0M [00:00<?, ?B/s]

## Task 2 — Conditional Image Colorization

In [7]:
def task2_detect_objects(pil_image):
    seg_map = segment_image(pil_image)
    present_ids = np.unique(seg_map)
    labels = sorted({ADE_LABELS[i] for i in present_ids if i in ADE_LABELS})
    return labels, seg_map

def _hex_or_name_to_rgb(color_str):
    import matplotlib.colors as mcolors
    try:
        return tuple(int(c * 255) for c in mcolors.to_rgb(color_str.strip()))
    except ValueError:
        return (128, 128, 128)  # fallback grey for unparseable input

def task2_colorize(pil_image, region_color_pairs):
    """region_color_pairs: list of (label_substring, color_name_or_hex)"""
    seg_map = segment_image(pil_image)
    img_rgb = np.array(pil_image.convert("RGB"))
    lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB).astype(np.int16)

    for label_substr, color_str in region_color_pairs:
        target_rgb = _hex_or_name_to_rgb(color_str)
        target_lab = cv2.cvtColor(
            np.uint8([[target_rgb]]), cv2.COLOR_RGB2LAB
        )[0, 0].astype(np.int16)

        matching_ids = [i for i, name in ADE_LABELS.items() if label_substr.lower() in name.lower()]
        mask = np.isin(seg_map, matching_ids)
        if not mask.any():
            continue
        lab[..., 1][mask] = target_lab[1]   # 'a' channel -> target hue
        lab[..., 2][mask] = target_lab[2]   # 'b' channel -> target hue
        # L (lightness) channel is untouched, so shading/texture survives

    lab = np.clip(lab, 0, 255).astype(np.uint8)
    result_rgb = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
    return Image.fromarray(result_rgb)


## Task 3 — Context-Aware Colorization of Complex Scenes

In [8]:
def _region_avg_color(lab, mask):
    if not mask.any():
        return None
    return lab[mask].mean(axis=0)

def task3_colorize(pil_image, region_color_pairs, apply_reflections=True, apply_shadows=True):

    base = task2_colorize(pil_image, region_color_pairs)
    seg_map = segment_image(pil_image)
    lab = cv2.cvtColor(np.array(base), cv2.COLOR_RGB2LAB).astype(np.int16)
    h, w = seg_map.shape

    sky_ids = [i for i, n in ADE_LABELS.items() if "sky" in n.lower()]
    building_ids = [i for i, n in ADE_LABELS.items() if "building" in n.lower()]
    reflective_ids = [i for i, n in ADE_LABELS.items() if "water" in n.lower() or "glass" in n.lower() or "mirror" in n.lower()]

    if apply_reflections and reflective_ids:
        source_mask = np.isin(seg_map, sky_ids + building_ids)
        source_color = _region_avg_color(lab, source_mask)
        if source_color is not None:
            refl_mask = np.isin(seg_map, reflective_ids)
            lower_half = np.zeros_like(refl_mask)
            lower_half[h // 2:, :] = True
            refl_mask = refl_mask & lower_half
            lab[..., 1][refl_mask] = (0.5 * lab[..., 1][refl_mask] + 0.5 * source_color[1]).astype(np.int16)
            lab[..., 2][refl_mask] = (0.5 * lab[..., 2][refl_mask] + 0.5 * source_color[2]).astype(np.int16)

    if apply_shadows:
        l_channel = lab[..., 0]
        threshold = np.percentile(l_channel, 15)
        shadow_mask = l_channel <= threshold
        lab[..., 1][shadow_mask] = (lab[..., 1][shadow_mask] * 0.7 + 128 * 0.3).astype(np.int16)  # desaturate
        lab[..., 2][shadow_mask] = np.clip(lab[..., 2][shadow_mask] - 6, 0, 255)  # cool/blue shift

    lab = np.clip(lab, 0, 255).astype(np.uint8)
    return Image.fromarray(cv2.cvtColor(lab, cv2.COLOR_LAB2RGB))


## Task 4 — Time-Based Historical Image Colorization

In [9]:
CLIP_ID = "openai/clip-vit-base-patch32"
_clip_model = CLIPModel.from_pretrained(CLIP_ID).to(device)
_clip_processor = CLIPProcessor.from_pretrained(CLIP_ID)

ERA_DEFINITIONS = {
    "1920s": {"prompt": "a photo from the 1920s, sepia tones", "sepia": 0.55, "saturation": 0.65},
    "WWII era (1940s)": {"prompt": "a photo from World War 2 era, muted olive and grey tones", "sepia": 0.25, "saturation": 0.55, "tint": (-10, 5)},
    "1950s": {"prompt": "a photo from the 1950s, pastel Kodachrome colors", "sepia": 0.0, "saturation": 1.15},
    "1970s": {"prompt": "a photo from the 1970s, warm faded film colors", "sepia": 0.15, "saturation": 0.85, "tint": (8, -4)},
    "Modern (2000s+)": {"prompt": "a modern digital photo, natural vivid colors", "sepia": 0.0, "saturation": 1.0},
}

def task4_detect_era(pil_image, manual_override=None):
    if manual_override and manual_override != "Auto-detect":
        return manual_override, 1.0
    labels = list(ERA_DEFINITIONS.keys())
    inputs = _clip_processor(text=labels, images=pil_image, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        logits = _clip_model(**inputs).logits_per_image.softmax(dim=1)[0]
    best_idx = int(logits.argmax())
    return labels[best_idx], float(logits[best_idx])

def _apply_era_grading(pil_image, era):
    cfg = ERA_DEFINITIONS[era]
    rgb = np.array(pil_image.convert("RGB")).astype(np.float32)
    gray = cv2.cvtColor(rgb.astype(np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float32)

    # saturation adjust in HSV
    hsv = cv2.cvtColor(rgb.astype(np.uint8), cv2.COLOR_RGB2HSV).astype(np.float32)
    hsv[..., 1] = np.clip(hsv[..., 1] * cfg["saturation"], 0, 255)
    rgb = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB).astype(np.float32)

    # sepia blend
    if cfg["sepia"] > 0:
        sepia = np.stack([gray * 1.07, gray * 0.94, gray * 0.73], axis=-1)
        rgb = rgb * (1 - cfg["sepia"]) + sepia * cfg["sepia"]

    # optional tint push (a,b-ish channel nudges via simple RGB bias)
    if "tint" in cfg:
        rgb[..., 0] += cfg["tint"][0]
        rgb[..., 2] += cfg["tint"][1]

    rgb = np.clip(rgb, 0, 255).astype(np.uint8)
    return Image.fromarray(rgb)

def task4_colorize(pil_image, manual_override=None, refine_with_diffusion=False, strength=0.35):
    era, confidence = task4_detect_era(pil_image, manual_override)
    graded = _apply_era_grading(pil_image, era)
    if refine_with_diffusion:
        try:
            ckpt = get_working_checkpoint(StableDiffusionImg2ImgPipeline)
            pipe = StableDiffusionImg2ImgPipeline.from_pretrained(ckpt, torch_dtype=dtype, safety_checker=None).to(device)
            graded = pipe(prompt=ERA_DEFINITIONS[era]["prompt"], image=graded, strength=strength).images[0]
        except Exception as e:
            print(f"[Task 4] Diffusion refinement skipped ({e}); returning deterministic grading only.")
    return graded, era, confidence


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

## Task 5 — Cross-Domain Image Colorization

In [10]:
def _infrared_false_color(pil_image):
    """Deterministic NIR-style false color: remap grayscale intensity bands to a
    red/green/blue false-color scheme, the way real infrared satellite products do."""
    gray = np.array(pil_image.convert("L")).astype(np.float32) / 255.0
    r = np.clip(gray * 1.4, 0, 1)          # vegetation/heat -> red-heavy
    g = np.clip((gray - 0.2) * 1.2, 0, 1)  # mid-intensity -> green
    b = np.clip(1 - gray, 0, 1)            # cold/water -> blue
    false_color = (np.stack([r, g, b], axis=-1) * 255).astype(np.uint8)
    return Image.fromarray(false_color)

def _xray_colormap(pil_image, cmap=cv2.COLORMAP_INFERNO):
    """Deterministic intensity colormap — preserves the scan's real values, adds no
    hallucinated detail. Appropriate for diagnostic-style imagery."""
    gray = np.array(pil_image.convert("L"))
    colored_bgr = cv2.applyColorMap(gray, cmap)
    colored_rgb = cv2.cvtColor(colored_bgr, cv2.COLOR_BGR2RGB)
    return Image.fromarray(colored_rgb)

DOMAIN_DEFINITIONS = {
    "Sketch": {"algorithm": "diffusion", "prompt": "a full color, richly detailed illustration", "strength": 0.6},
    "Infrared satellite": {"algorithm": "deterministic_infrared"},
    "Grayscale photograph": {"algorithm": "diffusion", "prompt": "a naturally colorized photograph", "strength": 0.45},
    "X-ray / medical scan": {"algorithm": "deterministic_colormap"},
}

def task5_colorize(pil_image, domain, strength_override=None):
    cfg = DOMAIN_DEFINITIONS[domain]
    if cfg["algorithm"] == "deterministic_infrared":
        return _infrared_false_color(pil_image)
    if cfg["algorithm"] == "deterministic_colormap":
        return _xray_colormap(pil_image)
    # generative path (sketch / grayscale photograph only)
    ckpt = get_working_checkpoint(StableDiffusionImg2ImgPipeline)
    pipe = StableDiffusionImg2ImgPipeline.from_pretrained(ckpt, torch_dtype=dtype, safety_checker=None).to(device)
    strength = strength_override if strength_override is not None else cfg["strength"]
    return pipe(prompt=cfg["prompt"], image=pil_image, strength=strength).images[0]


## Task 6 — Colorization of Historical Photographs (Fine-Tuned)

In [11]:
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import glob

os.makedirs("dataset/historical_color", exist_ok=True)
os.makedirs("checkpoints", exist_ok=True)

class PairedColorizationDataset(Dataset):
    """Builds grayscale-input / color-target PAIRS from real color images —
    this is what makes it an actual colorization training set, unlike the
    original text-to-image LoRA approach."""
    def __init__(self, image_dir, size=256):
        self.paths = sorted(glob.glob(os.path.join(image_dir, "*.jpg")) +
                             glob.glob(os.path.join(image_dir, "*.jpeg")) +
                             glob.glob(os.path.join(image_dir, "*.png")))
        self.size = size

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        color = Image.open(self.paths[idx]).convert("RGB").resize((self.size, self.size))
        color_np = np.array(color).astype(np.float32) / 255.0
        gray_np = cv2.cvtColor((color_np * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
        gray_tensor = torch.from_numpy(gray_np).unsqueeze(0)          # 1 x H x W  (input)
        color_tensor = torch.from_numpy(color_np).permute(2, 0, 1)    # 3 x H x W  (target)
        return gray_tensor, color_tensor


class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
            nn.Conv2d(out_c, out_c, 3, padding=1), nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)

class ColorizationUNet(nn.Module):
    """Small U-Net: grayscale (1ch) in, RGB (3ch) out. This is a genuine
    image-to-image colorization mapping, trained on paired data."""
    def __init__(self):
        super().__init__()
        self.enc1 = ConvBlock(1, 32)
        self.enc2 = ConvBlock(32, 64)
        self.enc3 = ConvBlock(64, 128)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = ConvBlock(128, 256)
        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec3 = ConvBlock(256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = ConvBlock(128, 64)
        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec1 = ConvBlock(64, 32)
        self.out_conv = nn.Conv2d(32, 3, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b = self.bottleneck(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return torch.sigmoid(self.out_conv(d1))


def train_task6_colorizer(image_dir="dataset/historical_color", epochs=15, batch_size=4, lr=2e-4):
    dataset = PairedColorizationDataset(image_dir)
    if len(dataset) < 5:
        raise ValueError(
            f"Only {len(dataset)} training images found in '{image_dir}'. "
            "Add more historical color photos to this folder before training "
            "(committed to the repo, not uploaded ad hoc at runtime)."
        )
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    model = ColorizationUNet().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        total_loss = 0.0
        for gray, color in loader:
            gray, color = gray.to(device), color.to(device)
            pred = model(gray)
            loss = F.l1_loss(pred, color)
            opt.zero_grad(); loss.backward(); opt.step()
            total_loss += loss.item()
        print(f"[Task 6] epoch {epoch+1}/{epochs}  loss={total_loss/len(loader):.4f}")

    weights_path = "checkpoints/colorizer_unet.pth"
    torch.save(model.state_dict(), weights_path)
    print(f"[Task 6] Saved weights to {weights_path}")
    return model

_task6_model = None
def load_task6_model(weights_path="checkpoints/colorizer_unet.pth"):
    global _task6_model
    if _task6_model is None:
        _task6_model = ColorizationUNet().to(device)
        _task6_model.load_state_dict(torch.load(weights_path, map_location=device))
        _task6_model.eval()
    return _task6_model

def task6_colorize(pil_image, era_period="Auto-detect"):
    model = load_task6_model()
    size = 256
    gray = np.array(pil_image.convert("L").resize((size, size))).astype(np.float32) / 255.0
    gray_tensor = torch.from_numpy(gray).unsqueeze(0).unsqueeze(0).to(device)
    with torch.no_grad():
        pred = model(gray_tensor)[0].permute(1, 2, 0).cpu().numpy()
    colorized = Image.fromarray((pred * 255).astype(np.uint8)).resize(pil_image.size)

    # period dropdown still does real work: it applies era-appropriate grading
    # on top of the network's learned colorization, using the Task 4 LUTs
    if era_period != "Auto-detect" and era_period in ERA_DEFINITIONS:
        colorized = _apply_era_grading(colorized, era_period)
    return colorized


In [12]:
from google.colab import files
import shutil, os

os.makedirs("dataset/historical_color", exist_ok=True)
print("Upload 8-10 COLOR photos (any color photos, doesn't need to be historical):")
uploaded = files.upload()
for fname in uploaded.keys():
    shutil.move(fname, f"dataset/historical_color/{fname}")
print(f"Total images in dataset: {len(os.listdir('dataset/historical_color'))}")

train_task6_colorizer(image_dir="dataset/historical_color", epochs=15)

Upload 8-10 COLOR photos (any color photos, doesn't need to be historical):


Saving Image 1.jpg to Image 1.jpg
Saving Image 2.jpg to Image 2.jpg
Saving Image 3.jpg to Image 3.jpg
Saving Image 4.jpg to Image 4.jpg
Saving Image 5.jpg to Image 5.jpg
Saving Image 6.jpg to Image 6.jpg
Saving Image 7.jpg to Image 7.jpg
Saving Image 7.webp to Image 7.webp
Saving Image 8.jpg to Image 8.jpg
Saving Image 9.jpg to Image 9.jpg
Saving Image 10.jpg to Image 10.jpg
Total images in dataset: 11
[Task 6] epoch 1/15  loss=0.2551
[Task 6] epoch 2/15  loss=0.2184
[Task 6] epoch 3/15  loss=0.2040
[Task 6] epoch 4/15  loss=0.1849
[Task 6] epoch 5/15  loss=0.1795
[Task 6] epoch 6/15  loss=0.1732
[Task 6] epoch 7/15  loss=0.1690
[Task 6] epoch 8/15  loss=0.1616
[Task 6] epoch 9/15  loss=0.1692
[Task 6] epoch 10/15  loss=0.1541
[Task 6] epoch 11/15  loss=0.1540
[Task 6] epoch 12/15  loss=0.1545
[Task 6] epoch 13/15  loss=0.1703
[Task 6] epoch 14/15  loss=0.1498
[Task 6] epoch 15/15  loss=0.1518
[Task 6] Saved weights to checkpoints/colorizer_unet.pth


ColorizationUNet(
  (enc1): ConvBlock(
    (net): Sequential(
      (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    )
  )
  (enc2): ConvBlock(
    (net): Sequential(
      (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    )
  )
  (enc3): ConvBlock(
    (net): Sequential(
      (0): Conv2d(64, 128, kernel_size=

## Combined Gradio App — all tasks in one interface

In [13]:
with gr.Blocks(title="Real-Time Gen AI Image Colorization)") as demo:
    gr.Markdown("# Real-Time Gen AI Image Colorization — Corrected Submission")

    with gr.Tab("Base: Text-to-Image Generator"):
        prompt_in = gr.Textbox(label="Prompt")
        model_choice = gr.Dropdown(list(MODEL_MAP.keys()), value="Stable Diffusion 1.5 (base)", label="Model")
        gen_btn = gr.Button("Generate")
        gen_out = gr.Image(label="Result")
        gen_btn.click(lambda p, m: base_generator.generate(p, m), [prompt_in, model_choice], gen_out)

    with gr.Tab("Task 1: Real-time Multi-Object Colorization"):
        with gr.Row():
            with gr.Column():
                gr.Markdown("**Live webcam**")
                webcam_in = gr.Image(sources=["webcam"], streaming=True, label="Webcam")
                webcam_out = gr.Image(label="Colorized")
                webcam_in.stream(process_webcam_frame, webcam_in, webcam_out)
            with gr.Column():
                gr.Markdown("**Upload a video**")
                video_in = gr.Video(label="Upload video")
                video_btn = gr.Button("Colorize video (every frame)")
                video_out = gr.Video(label="Colorized output")
                video_btn.click(process_video_file, video_in, video_out)

    with gr.Tab("Task 2: Conditional Colorization"):
        t2_img = gr.Image(type="pil", label="Grayscale photo")
        t2_detect_btn = gr.Button("Detect regions")
        t2_labels_out = gr.Textbox(label="Detected regions", interactive=False)
        t2_regions = gr.Textbox(label="Region:color pairs, one per line, e.g.\nsky: lightblue\ngrass: green")
        t2_run_btn = gr.Button("Colorize (mask-guided)")
        t2_out = gr.Image(label="Result")

        def t2_detect(img):
            labels, _ = task2_detect_objects(img)
            return ", ".join(labels)
        t2_detect_btn.click(t2_detect, t2_img, t2_labels_out)

        def t2_run(img, region_text):
            pairs = []
            for line in region_text.strip().splitlines():
                if ":" in line:
                    label, color = line.split(":", 1)
                    pairs.append((label.strip(), color.strip()))
            return task2_colorize(img, pairs)
        t2_run_btn.click(t2_run, [t2_img, t2_regions], t2_out)

    with gr.Tab("Task 3: Context-Aware Complex Scenes"):
        t3_img = gr.Image(type="pil", label="Photo")
        t3_regions = gr.Textbox(label="Region:color pairs (same format as Task 2)")
        t3_reflections = gr.Checkbox(value=True, label="Apply reflection heuristic")
        t3_shadows = gr.Checkbox(value=True, label="Apply shadow heuristic")
        t3_btn = gr.Button("Colorize")
        t3_out = gr.Image(label="Result")

        def t3_run(img, region_text, refl, shad):
            pairs = []
            for line in region_text.strip().splitlines():
                if ":" in line:
                    label, color = line.split(":", 1)
                    pairs.append((label.strip(), color.strip()))
            return task3_colorize(img, pairs, refl, shad)
        t3_btn.click(t3_run, [t3_img, t3_regions, t3_reflections, t3_shadows], t3_out)

    with gr.Tab("Task 4: Historical Era Colorization"):
        t4_img = gr.Image(type="pil", label="Grayscale photo")
        t4_override = gr.Dropdown(["Auto-detect"] + list(ERA_DEFINITIONS.keys()), value="Auto-detect", label="Era (manual override)")
        t4_refine = gr.Checkbox(value=False, label="Refine with diffusion")
        t4_btn = gr.Button("Colorize")
        t4_out = gr.Image(label="Result")
        t4_era_out = gr.Textbox(label="Detected era", interactive=False)

        def t4_run(img, override, refine):
            result, era, conf = task4_colorize(img, override, refine)
            return result, f"{era} (confidence {conf:.2f})"
        t4_btn.click(t4_run, [t4_img, t4_override, t4_refine], [t4_out, t4_era_out])

    with gr.Tab("Task 5: Cross-Domain Colorization"):
        t5_img = gr.Image(type="pil", label="Input image")
        t5_domain = gr.Dropdown(list(DOMAIN_DEFINITIONS.keys()), label="Input domain")
        t5_strength = gr.Slider(0.1, 0.9, value=0.5, label="Strength (generative domains only)")
        t5_btn = gr.Button("Colorize")
        t5_out = gr.Image(label="Result")
        t5_btn.click(task5_colorize, [t5_img, t5_domain, t5_strength], t5_out)

    with gr.Tab("Task 6: Historical Photographs"):
        gr.Markdown(
            "Training data lives in `dataset/historical_color/` (committed to the repo). "
            "Run `train_task6_colorizer()` once in a code cell before using this tab."
        )
        t6_img = gr.Image(type="pil", label="Grayscale historical photo")
        t6_period = gr.Dropdown(["Auto-detect"] + list(ERA_DEFINITIONS.keys()), value="Auto-detect", label="Period")
        t6_btn = gr.Button("Colorize")
        t6_out = gr.Image(label="Result")
        t6_btn.click(task6_colorize, [t6_img, t6_period], t6_out)

demo.launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a2048a6f8366032c69.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.13/dist-packages/gradio/routes.py:1368: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://a2048a6f8366032c69.gradio.live
